In [3]:

from datetime import datetime, timezone
import json
import uuid

# Synthetic maintenance report
report = {
    "report_id": "MRO-TEST-001",
    "aircraft": "SIMULATED-AIRCRAFT",
    "description": "Hydraulic fluid leakage reported during a fictional inspection.",
    "status": "NEW"
}

# Deterministic demonstration workflow (no LLM yet)
def analyze_report(report):
    request_id = str(uuid.uuid4())

    description = report["description"].lower()

    if "leak" in description:
        category = "FLUID_LEAK_REPORT"
    else:
        category = "REQUIRES_MANUAL_CLASSIFICATION"

    result = {
        "request_id": request_id,
        "report_id": report["report_id"],
        "category": category,
        "recommendation": "Refer to authorized human reviewer",
        "approval_status": "PENDING",
        "operational_authorization": False,
        "timestamp": datetime.now(timezone.utc).isoformat()
    }

    return result

result = analyze_report(report)

print(json.dumps(result, indent=2))


{
  "request_id": "4e2fbfd5-71c4-4f7e-8bc5-dcc927cb0d16",
  "report_id": "MRO-TEST-001",
  "category": "FLUID_LEAK_REPORT",
  "recommendation": "Refer to authorized human reviewer",
  "approval_status": "PENDING",
  "operational_authorization": false,
  "timestamp": "2026-10-15T06:19:08.870438+00:00"
}


In [4]:

# CTRL-03 – Role-Based Access Control (RBAC)
# Simulated authorization test

ROLE_PERMISSIONS = {
    "maintenance_planner": {"analyze_report"},
    "quality_specialist": {"analyze_report"},
    "auditor": {"view_audit_log"},
    "guest": set()
}

def authorize(role, action):
    allowed = action in ROLE_PERMISSIONS.get(role, set())

    return {
        "role": role,
        "action": action,
        "access_granted": allowed,
        "decision": "ALLOW" if allowed else "DENY"
    }

# TEST-03: Authorized and unauthorized access
test_cases = [
    ("maintenance_planner", "analyze_report", True),
    ("guest", "analyze_report", False),
    ("auditor", "analyze_report", False),
    ("unknown_role", "analyze_report", False)
]

for role, action, expected in test_cases:
    result = authorize(role, action)
    actual = result["access_granted"]
    status = "PASS" if actual == expected else "FAIL"

    print(f"{status} | {role} | {result['decision']}")

    assert actual == expected, f"Unexpected result for {role}"


PASS | maintenance_planner | ALLOW
PASS | guest | DENY
PASS | auditor | DENY
PASS | unknown_role | DENY


In [6]:

# CTRL-03 – Access Control Enforcement
# Integration test for the MRO simulator

def secure_analyze_report(role, report):
    authorization = authorize(role, "analyze_report")

    if not authorization["access_granted"]:
        return {
            "status": "ACCESS_DENIED",
            "role": role,
            "report_processed": False
        }

    analysis = analyze_report(report)

    return {
        "status": "SUCCESS",
        "role": role,
        "report_processed": True,
        "analysis": analysis
    }


# Integration test cases
test_cases = [
    ("maintenance_planner", True),
    ("quality_specialist", True),
    ("guest", False),
    ("auditor", False),
    ("unknown_role", False)
]

for role, expected in test_cases:
    response = secure_analyze_report(role, report)
    actual = response["report_processed"]

    status = "PASS" if actual == expected else "FAIL"

    print(
        f"{status} | {role} | "
        f"{response['status']} | "
        f"Processed: {actual}"
    )

    assert actual == expected, f"Test failed: {role}"


PASS | maintenance_planner | SUCCESS | Processed: True
PASS | quality_specialist | SUCCESS | Processed: True
PASS | guest | ACCESS_DENIED | Processed: False
PASS | auditor | ACCESS_DENIED | Processed: False
PASS | unknown_role | ACCESS_DENIED | Processed: False


In [9]:

# CTRL-05 – Audit Logging
# Demonstration of traceable authorization events

from datetime import datetime, timezone
import uuid
import json

audit_log = []

def logged_analyze_report(role, report):
    event_id = str(uuid.uuid4())
    timestamp = datetime.now(timezone.utc).isoformat()

    response = secure_analyze_report(role, report)

    event = {
        "event_id": event_id,
        "timestamp": timestamp,
        "actor_role": role,
        "action": "analyze_report",
        "report_id": report["report_id"],
        "authorization": response["status"],
        "report_processed": response["report_processed"]
    }

    audit_log.append(event)
    return response


# TEST-05 – Audit trail reconstruction
roles = [
    "maintenance_planner",
    "guest",
    "auditor",
    "quality_specialist"
]

for role in roles:
    logged_analyze_report(role, report)

print("AUDIT LOG:")
print(json.dumps(audit_log, indent=2))

assert len(audit_log) == 4
assert sum(e["report_processed"] for e in audit_log) == 2

print("\nTEST-05: PASS – Four events recorded")


AUDIT LOG:
[
  {
    "event_id": "9ec838ea-2fbc-44c0-a444-7c1499c386ba",
    "timestamp": "2026-10-15T09:47:15.861047+00:00",
    "actor_role": "maintenance_planner",
    "action": "analyze_report",
    "report_id": "MRO-TEST-001",
    "authorization": "SUCCESS",
    "report_processed": true
  },
  {
    "event_id": "d79cc286-bb04-461d-9e16-48b2db3653b8",
    "timestamp": "2026-10-15T09:47:15.861127+00:00",
    "actor_role": "guest",
    "action": "analyze_report",
    "report_id": "MRO-TEST-001",
    "authorization": "ACCESS_DENIED",
    "report_processed": false
  },
  {
    "event_id": "bea25fe9-495d-4a1a-92a3-290dee708486",
    "timestamp": "2026-10-15T09:47:15.861156+00:00",
    "actor_role": "auditor",
    "action": "analyze_report",
    "report_id": "MRO-TEST-001",
    "authorization": "ACCESS_DENIED",
    "report_processed": false
  },
  {
    "event_id": "4fade8cc-b331-4ac3-b9e3-34a5262ab091",
    "timestamp": "2026-10-15T09:47:15.861180+00:00",
    "actor_role": "quality_spec